In [37]:
import pandas as pd

In [38]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [39]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [40]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Text PreProcessing

### 1. Convert to Lowercase

In [41]:
df["review"] = df["review"].str.lower() 

### 2. Remove URLs

In [42]:
import re

In [43]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "" , text)
    return text

In [44]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [45]:
df["review"] = df["review"].apply(remove_urls)

### 3. Remove Punctuation

In [46]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text

In [47]:
df["review"] = df["review"].apply(remove_punctuation)

### 4. Remove HTML Tags

In [48]:
def remove_tags(text):
    text = re.sub(r"<.*?>" , "" , text)
    return text

In [49]:
df["review"] = df["review"].apply(remove_tags)

### 5. Remove Stopwords

In [50]:
import nltk

In [51]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/bat-linux/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/bat-
[nltk_data]     linux/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/bat-
[nltk_data]     linux/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [52]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [53]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text

In [54]:
df["review"] = df["review"].apply(remove_stopwords)

In [55]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [56]:
from nltk.stem import PorterStemmer

In [57]:
def stemming(text): 
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens: 
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)

In [58]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [60]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder() 
df["sentiment"] = le.fit_transform(df["sentiment"])

In [62]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [64]:
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [65]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Datasets and DataLoaders

In [66]:
from sklearn.model_selection import train_test_split

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [70]:
import torch 
from torch.utils.data import TensorDataset , DataLoader

In [69]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [73]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [74]:
train_loader = DataLoader(
    train_set,
    shuffle=True, 
    batch_size=64
)

test_loader = DataLoader(
    test_set,
    shuffle=True, 
    batch_size=64
)

## Build our RNN

In [78]:
import torch.nn as nn
import torch.optim as optim

In [77]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size = 128 , num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        # optional => hidden_state shape (no. of layers , batch_size , hidden_size)
        h0 = torch.zeros(self.num_layers , x.size(0) , self.hidden_size)

        out,_ = self.rnn(x,h0)
        # 1st value = hidden state of all the timestamps
        # 2nd value = final hidden state of last timestep 

        out = self.fc(out[:,-1, :])
        return out

In [79]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [80]:
epochs = 10

for epoch in range(epochs): 
    model.train()

    for Xb,yb in train_loader: 
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)    # add singleton direction

        outputs = model(Xb)  # (batch_size,1)
        outputs = torch.sigmoid(outputs.squeeze())   # (batch_size , ) => probability

        loss = criterion(outputs,yb)
        loss.backward()

        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/10 and loss = 0.35969799757003784
epoch = 2/10 and loss = 0.28406497836112976
epoch = 3/10 and loss = 0.15343524515628815
epoch = 4/10 and loss = 0.1812157928943634
epoch = 5/10 and loss = 0.25708645582199097
epoch = 6/10 and loss = 0.29378119111061096
epoch = 7/10 and loss = 0.4210973381996155
epoch = 8/10 and loss = 0.24657440185546875
epoch = 9/10 and loss = 0.11853551119565964
epoch = 10/10 and loss = 0.1769101321697235


## Evaluation

In [81]:
model.eval()

with torch.no_grad(): 
    correct_out = 0
    total_out = 0

    for Xb,yb in test_loader: 
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_out += yb.size(0) 
        correct_out += (predicted == yb).sum().item()

    print(f"Accuracy = {(correct_out/total_out)*100}")

Accuracy = 85.6609861853383
